# RuleShift Notebook Test

This notebook keeps the official RuleShift path short: attach the packaged runtime, load the frozen leaderboard split, run the benchmark, review the result, and inspect the audit tables only if needed.

## Setup

Run the next two cells to make the packaged runtime importable and load the frozen leaderboard data used by the benchmark.

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/datasets/raptorengineer/ruleshift-runtime/src — "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

In [ ]:
import pandas as pd

from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)
from core.splits import MANIFEST_VERSION

PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe()

## Status

This quick check confirms the benchmark version, loaded episodes, available splits, and whether the private dataset is attached before you run the task.

In [ ]:
_status_df = pd.DataFrame(
    [
        ("Benchmark version", MANIFEST_VERSION),
        ("Episodes loaded", len(leaderboard_df)),
        ("Splits available", ", ".join(sorted(frozen_splits))),
        ("Private dataset", "yes" if PRIVATE_DATASET_ROOT is not None else "no"),
    ],
    columns=["Field", "Value"],
)

_status_df

## Official task

`ruleshift_benchmark_v1_binary` is the only leaderboard-facing task. It returns only `(numerator, denominator)`; the notebook keeps `_RULESHIFT_PAYLOAD` and `_RULESHIFT_BINARY_DF` for post-run inspection.

In [ ]:
_RULESHIFT_BINARY_DF = None
_RULESHIFT_PAYLOAD = None
_RULESHIFT_OFFICIAL_RESULT = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary_row",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
    store_task=False,
)
def _ruleshift_benchmark_v1_binary_row(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


def _run_ruleshift_benchmark_v1_binary_eval(llm):
    eval_df = leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]].copy()
    binary_results = _ruleshift_benchmark_v1_binary_row.evaluate(
        llm=[llm],
        evaluation_data=eval_df,
    )
    binary_df = normalize_count_result_df(
        binary_results.as_dataframe(),
        dataframe_label="binary_df",
    )
    payload = build_kaggle_payload(binary_df=binary_df)
    validate_kaggle_payload(payload)
    official_result = (payload["numerator"], payload["denominator"])
    return binary_df, payload, official_result


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of marker records, "
        "then predict four post-shift probe outputs."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> tuple[int, int]:
    import json

    global _RULESHIFT_BINARY_DF, _RULESHIFT_PAYLOAD, _RULESHIFT_OFFICIAL_RESULT

    binary_df, payload, official_result = _run_ruleshift_benchmark_v1_binary_eval(llm)

    _RULESHIFT_BINARY_DF = binary_df
    _RULESHIFT_PAYLOAD = payload
    _RULESHIFT_OFFICIAL_RESULT = official_result

    print("=== RuleShift Notebook Test — Canonical Result ===")
    print(json.dumps(payload, indent=2))
    print("=== End Canonical Result ===")
    return official_result


## Run benchmark

Run this cell to evaluate the benchmark and capture the official payload.

In [ ]:
score = ruleshift_benchmark_v1_binary(kbench.llm)
payload = _RULESHIFT_PAYLOAD
if payload is None:
    raise RuntimeError("ruleshift_benchmark_v1_binary did not populate _RULESHIFT_PAYLOAD")

## Run summary

The next cells show the canonical result first, then a compact per-episode summary if there were any misses.

In [ ]:
_score_pct = f"{payload['score'] * 100:.1f}%"
_num = payload["numerator"]
_den = payload["denominator"]
_episodes = payload["total_episodes"]
_version = payload["benchmark_version"]

_result_df = pd.DataFrame(
    [
        ("Final score", _score_pct),
        ("Correct / Total probes", f"{_num} / {_den}"),
        ("Episodes evaluated", _episodes),
        ("Payload validation", "Passed"),
        ("Benchmark version", _version),
    ],
    columns=["Field", "Value"],
).set_index("Field")

_result_df

In [ ]:
_bdf = _RULESHIFT_BINARY_DF
_has_misses = (
    _bdf is not None
    and {"num_correct", "total"}.issubset(_bdf.columns)
    and (_bdf["num_correct"] < _bdf["total"]).any()
)

if _has_misses:
    _m = _bdf[_bdf["num_correct"] < _bdf["total"]].copy()
    _id_cols = [c for c in ["episode_id", "split"] if c in _m.columns]
    _summary_df = _m.loc[:, _id_cols].copy()
    _summary_df["correct"] = _m["num_correct"].astype(int)
    _summary_df["total"] = _m["total"].astype(int)
    _summary_df["missed"] = (_m["total"] - _m["num_correct"]).astype(int)
else:
    _rows = []
    for _sn, _feps in frozen_splits.items():
        for _fse in _feps:
            _rows.append((_sn.replace("_leaderboard", ""), _fse.episode.difficulty.name.lower()))
    _comp = pd.DataFrame(_rows, columns=["split", "difficulty"])
    _summary_df = _comp.groupby("split")["difficulty"].value_counts().unstack(fill_value=0)
    _order = [c for c in ["easy", "medium", "hard"] if c in _summary_df.columns]
    _summary_df = _summary_df[[*_order, *[c for c in _summary_df.columns if c not in _order]]]
    _summary_df["total"] = _summary_df.sum(axis=1)
    _summary_df = _summary_df.reset_index()

_summary_df

## Audit views

These diagnostics are optional. The cell below builds the audit tables after the main run so the notebook stays focused on setup, execution, and result review.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    from core.kaggle import build_audit_balance, build_audit_catalog, build_audit_failures
except ImportError:
    def build_audit_catalog(public_records):
        rows = []
        for record in public_records:
            if record.partition != "public_leaderboard":
                raise ValueError("build_audit_catalog expects only public_leaderboard records")
            episode = record.episode
            rows.append(
                {
                    "episode_id": episode.episode_id,
                    "split": record.partition,
                    "difficulty": episode.difficulty.value,
                    "transition": episode.transition.value,
                    "template_family": episode.template_family.value,
                    "template_id": episode.template_id.value,
                    "difficulty_profile_id": episode.difficulty_profile_id.value,
                    "pre_count": episode.pre_count,
                    "post_labeled_count": episode.post_labeled_count,
                    "contradiction_count_post": episode.contradiction_count_post,
                }
            )

        audit_catalog = pd.DataFrame(
            rows,
            columns=[
                "episode_id",
                "split",
                "difficulty",
                "transition",
                "template_family",
                "template_id",
                "difficulty_profile_id",
                "pre_count",
                "post_labeled_count",
                "contradiction_count_post",
            ],
        )
        if audit_catalog["episode_id"].duplicated().any():
            raise ValueError("audit catalog episode_id values must be unique")
        return audit_catalog

    def build_audit_balance(audit_catalog):
        required_columns = {
            "episode_id",
            "split",
            "difficulty",
            "transition",
            "template_family",
            "template_id",
            "difficulty_profile_id",
            "pre_count",
            "post_labeled_count",
            "contradiction_count_post",
        }
        missing = required_columns - set(audit_catalog.columns)
        if missing:
            raise ValueError(f"audit_catalog is missing required columns: {sorted(missing)}")

        views = []
        for dimension in ("difficulty", "transition", "template_family", "template_id"):
            dimension_view = (
                audit_catalog.groupby(dimension, dropna=False)
                .size()
                .reset_index(name="episodes")
                .rename(columns={dimension: "value"})
            )
            dimension_view.insert(0, "dimension", dimension)
            views.append(dimension_view[["dimension", "value", "episodes"]])

        if not views:
            return pd.DataFrame(columns=["dimension", "value", "episodes"])

        balance = pd.concat(views, ignore_index=True)
        return balance.sort_values(["dimension", "value"], ignore_index=True)

    def build_audit_failures(binary_df, audit_catalog):
        required_catalog_columns = {
            "episode_id",
            "split",
            "difficulty",
            "transition",
            "template_family",
            "template_id",
            "difficulty_profile_id",
            "pre_count",
            "post_labeled_count",
            "contradiction_count_post",
        }
        missing_catalog = required_catalog_columns - set(audit_catalog.columns)
        if missing_catalog:
            raise ValueError(f"audit_catalog is missing required columns: {sorted(missing_catalog)}")

        required_result_fields = {"episode_id", "split", "num_correct", "total"}
        missing_results = required_result_fields - set(binary_df.columns)
        if missing_results:
            raise ValueError(f"binary_df is missing required columns: {sorted(missing_results)}")

        public_results = binary_df[binary_df["split"] == "public_leaderboard"].copy()
        if public_results.empty:
            return pd.DataFrame(
                columns=[
                    "episode_id",
                    "split",
                    "difficulty",
                    "transition",
                    "template_family",
                    "template_id",
                    "num_correct",
                    "total",
                    "missed",
                ]
            )

        public_results["missed"] = public_results["total"] - public_results["num_correct"]
        failures = public_results[public_results["missed"] > 0].copy()
        if failures.empty:
            return pd.DataFrame(
                columns=[
                    "episode_id",
                    "split",
                    "difficulty",
                    "transition",
                    "template_family",
                    "template_id",
                    "num_correct",
                    "total",
                    "missed",
                ]
            )

        merged = failures.merge(
            audit_catalog,
            how="inner",
            on=["episode_id", "split"],
            validate="one_to_one",
        )
        result = merged[
            [
                "episode_id",
                "split",
                "difficulty",
                "transition",
                "template_family",
                "template_id",
                "num_correct",
                "total",
                "missed",
            ]
        ].copy()
        return result.sort_values(["missed", "episode_id"], ascending=[False, True], ignore_index=True)

audit_catalog = build_audit_catalog(frozen_splits["public_leaderboard"])
audit_balance = build_audit_balance(audit_catalog)
audit_failures = build_audit_failures(_RULESHIFT_BINARY_DF, audit_catalog)

display(audit_catalog)
display(audit_balance)
audit_failures

## Task selection

Use the final cell to select the official leaderboard entry point.

In [ ]:
%choose ruleshift_benchmark_v1_binary